In [1]:
# Setup
import sys
sys.path.append('..')

from src.vector_store import VectorStore
from src.rag_chain import RAGChain
from src.config import Config
import logging

# Enable logging to see RAG pipeline steps
logging.basicConfig(level=logging.INFO, format='%(message)s')

print("✓ Imports successful")

✓ Imports successful


## Step 1: Initialize RAG System

In [2]:
# Load configuration
config = Config()

# Check prerequisites
if Config.AZURE_OPENAI_ENDPOINT:
    print("✓ Configuration loaded (Azure OpenAI)")
    print(f"  Endpoint: {Config.AZURE_OPENAI_ENDPOINT}")
    print(f"  Chat deployment: {Config.AZURE_OPENAI_CHAT_DEPLOYMENT}")
    print(f"  Qdrant: {config.qdrant_url}")
    print(f"  Collection: {config.qdrant_collection}")
elif Config.OPENAI_API_KEY:
    print("✓ Configuration loaded (Standard OpenAI)")
    print(f"  Qdrant: {config.qdrant_url}")
    print(f"  Collection: {config.qdrant_collection}")
else:
    print("⚠ No API key found. RAG system requires API access.")


✓ Configuration loaded (Azure OpenAI)
  Endpoint: https://initiative.openai.azure.com/
  Chat deployment: gpt-4.1
  Qdrant: http://localhost:6333
  Collection: confidential_docs


In [3]:
# Initialize vector store
if Config.AZURE_OPENAI_ENDPOINT:
    vector_store = VectorStore(
        qdrant_url=config.qdrant_url,
        collection_name=config.qdrant_collection,
        openai_api_key=Config.AZURE_OPENAI_API_KEY,
        azure_endpoint=Config.AZURE_OPENAI_ENDPOINT,
        api_version=Config.AZURE_OPENAI_API_VERSION,
        embedding_deployment=Config.AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    )
else:
    vector_store = VectorStore(
        qdrant_url=config.qdrant_url,
        collection_name=config.qdrant_collection,
        openai_api_key=Config.OPENAI_API_KEY
    )

# Check health
health = vector_store.health_check()
print(f"✓ Vector store connected: {health['status']}")

# Check collection
stats = vector_store.get_statistics()
print(f"  Elements in collection: {stats['points_count']:,}")

if stats['points_count'] == 0:
    print("\n⚠ Collection is empty. Please run 04_qdrant_storage.ipynb first to populate data.")


HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
✓ Connected to Qdrant at http://localhost:6333
✓ Embedding model: text-embedding-3-large (3072 dims)
HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"
HTTP Request: GET http://localhost:6333/collections/confidential_docs "HTTP/1.1 200 OK"


✓ Vector store connected: healthy
  Elements in collection: 124,776


In [4]:
# Initialize RAG Chain with Azure OpenAI (gpt-4.1 — best model for Q&A)
if Config.AZURE_OPENAI_ENDPOINT:
    rag_chain = RAGChain(
        vector_store=vector_store,
        openai_api_key=Config.AZURE_OPENAI_API_KEY,
        azure_endpoint=Config.AZURE_OPENAI_ENDPOINT,
        api_version=Config.AZURE_OPENAI_API_VERSION,
        azure_deployment=Config.AZURE_OPENAI_CHAT_DEPLOYMENT  # gpt-4.1
    )
    print("✓ RAG Chain initialized (Azure OpenAI)")
    print(f"  Deployment: {Config.AZURE_OPENAI_CHAT_DEPLOYMENT}")
    print("  Model: gpt-4.1")
else:
    rag_chain = RAGChain(
        vector_store=vector_store,
        openai_api_key=Config.OPENAI_API_KEY,
        model="gpt-4.1",
        temperature=0.3
    )
    print("✓ RAG Chain initialized (Standard OpenAI)")
    print("  Model: gpt-4.1")
print("  Temperature: 0.3")


✓ RAG Chain initialized with model: gpt-4.1


✓ RAG Chain initialized (Azure OpenAI)
  Deployment: gpt-4.1
  Model: gpt-4.1
  Temperature: 0.3


## Step 2: Simple Query

In [5]:
# Ask a simple question
query = "What are the main topics covered in these documents?"

print(f"\nQUERY: {query}\n")

result = rag_chain.query(
    user_query=query,
    top_k=5,
    include_related=True
)

# Format and display
print(rag_chain.format_response(result))


QUERY: What are the main topics covered in these documents?

Step 1: Retrieving relevant documents...
Generating embeddings for 1 texts...



QUERY: What are the main topics covered in these documents?



Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.84s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "H

ANSWER
Based on the provided context, the main topics covered in these documents are as follows:

1. **FLOWCAL Release Notes**  
   - Document: FLOWCAL-10.0.1.0 Release Notes.pdf  
   - Content: This document provides release notes for FLOWCAL version 10.0.1.0. It likely details new features, enhancements, bug fixes, and other updates related to this software version.

2. **Master Data Creation**  
   - Document: 05.01.02 Master Data Creation.docx  
   - Content: This document covers procedures and guidelines for creating master data. It includes both text and images, suggesting step-by-step instructions or visual aids for the master data creation process.

3. **Test Cases**  
   - Documents:  
     - 1671147 Test Case.docx  
     - Test Case v.2.docx  
     - 1360959 Test Case.docx  
   - Content: These documents contain test cases, which are likely used for validating software functionality or business processes. Each test case document may focus on different scenarios or requirement

## Step 3: Detailed Query with Context

In [ ]:
# Ask a more specific question
query = "Explain the methodology or approach for testing NHC Rollups test cases."

print(f"\nQUERY: {query}\n")

result = rag_chain.query(
    user_query=query,
    top_k=7,
    include_related=True,
    max_context_length=6000
)

print(rag_chain.format_response(result))


QUERY: Explain the methodology or approach for testing NHC Rollups test

Step 1: Retrieving relevant documents...
Generating embeddings for 1 texts...



QUERY: Explain the methodology or approach for testing NHC Rollups test



Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "H

ANSWER
Based on the provided documentation, the methodology for testing NHC Rollups involves several structured steps, as outlined in the relevant documents, particularly "Test Setup.docx" and "Location Rollup Test Data Setup.docx." Here is a comprehensive explanation:

---

### 1. **Test Data Preparation**
- **Location Rollup Test Data Setup.docx** provides guidance on preparing the necessary test data for NHC Rollups. This includes:
  - Creating and configuring location hierarchies required for rollup scenarios.
  - Ensuring that all relevant data points (such as locations, sub-locations, and associated attributes) are correctly set up in the test environment.
  - There are related images in this document that visually demonstrate the data setup process, which can help clarify the configuration steps.

### 2. **Test Environment Configuration**
- **Test Setup.docx** details the steps for configuring the test environment:
  - Setting up the system or application to ensure it reflects t

## Step 4: Multi-Modal Query (Images/Tables)

In [7]:
# Ask about visual content
query = "Are there any diagrams, charts, or tables? What do they show?"

print(f"\nQUERY: {query}\n")

result = rag_chain.query(
    user_query=query,
    top_k=5,
    include_related=True
)

print(rag_chain.format_response(result))


QUERY: Are there any diagrams, charts, or tables? What do they show?

Step 1: Retrieving relevant documents...
Generating embeddings for 1 texts...



QUERY: Are there any diagrams, charts, or tables? What do they show?



Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
  Retrieved 5 main results
  Retrieved 0 related elements

Step 2: Assembling context...
  Context length: 584 chars

Step 3: Generating answer...
HTTP Request: POST https://initiative.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"

Step 4: Formatting sources...

ANSWER GENERATED
Confidence: 62.10%
Sources: 5



ANSWER
Based on the provided context, there are several images referenced, which likely contain diagrams, charts, or tables. Here is a summary of what each image appears to show:

1. **AGA-3 1992_603x729.png**  
   - This image is likely related to the AGA-3 standard, which typically includes diagrams or tables about gas flow measurement. However, the exact content is not specified in the context.

2. **Meter Characteristic Fields Displayed.png** and **Meter Characteristic Fields Displayed_675x993.png**  
   - Both images suggest the display of meter characteristic fields, which usually involve tables or charts showing various parameters or properties of meters.

3. **Legend Alarm State Codes - 8.11.png**  
   - This image likely contains a legend or table explaining alarm state codes, which would help users interpret alarm indicators or statuses in a system.

4. **Liquid Flow Data Tab - 8.11.png**  
   - This image probably shows a tabular display of liquid flow data, which could incl

## Step 5: Examine Retrieved Context

In [8]:
# Query and inspect what was retrieved
query = "What are the key findings or results?"

print(f"\nQUERY: {query}\n")

result = rag_chain.query(
    user_query=query,
    top_k=5,
    include_related=True
)

# Show what was retrieved
print("=" * 80)
print("RETRIEVED CONTEXT")
print("=" * 80)

main_results = result['main_results']
related_elements = result['related_elements']

print(f"\nMain results: {len(main_results)}")
for idx, res in enumerate(main_results, 1):
    print(f"\n{idx}. {res['filename']} ({res['type']})")
    print(f"   Score: {res['score']:.3f}")
    print(f"   Path: {res.get('hierarchy_path', 'N/A')}")
    
    summary = res.get('summary_medium', '')
    if summary:
        snippet = summary[:200] + "..." if len(summary) > 200 else summary
        print(f"   Content: {snippet}")

print(f"\nRelated elements: {len(related_elements)}")
if related_elements:
    for idx, elem in enumerate(related_elements[:3], 1):
        print(f"  {idx}. {elem['type']} from {elem['filename']}")

# Show the answer
print("\n" + "=" * 80)
print("GENERATED ANSWER")
print("=" * 80)
print(f"\n{result['answer']}")
print(f"\nConfidence: {result['confidence']:.1%}")


QUERY: What are the key findings or results?

Step 1: Retrieving relevant documents...
Generating embeddings for 1 texts...



QUERY: What are the key findings or results?



Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "H

RETRIEVED CONTEXT

Main results: 5

1. 06.05.01.01 - 06.05.01.157 - Public API Performance TC.docx (text)
   Score: 0.508
   Path: 06.05.01.01 - 06.05.01.157 - Public API Performance TC.docx

2. 08.03.02.01 - Performance Benchmark.docx (text)
   Score: 0.508
   Path: 08.03.02.01 - Performance Benchmark.docx

3. 1135537 Performance TC - R1030.docx (text)
   Score: 0.508
   Path: 1135537 Performance TC - R1030.docx

4. 1135538 Performance TC for standard and custom RPTs - R1030 IT.docx (text)
   Score: 0.508
   Path: 1135538 Performance TC for standard and custom RPTs - R1030 IT.docx

5. 1119724 Performance TC for standard and custom RPTs - Dev FT.docx (text)
   Score: 0.508
   Path: 1119724 Performance TC for standard and custom RPTs - Dev FT.docx

Related elements: 20
  1. text from 06.05.01.01 - 06.05.01.157 - Public API Performance TC.docx
  2. text from 1135537 Performance TC - R1030.docx
  3. text from 1135537 Performance TC - R1030.docx

GENERATED ANSWER

Based on the provided con

## Step 6: Interactive Query Session

In [13]:
# Test multiple related queries
queries = [
    "Explain the methodology or approach for testing NHC Rollups test cases.?",
    "Which areas will be affected when any feature is implemented for NHC Rollups?",
    "Provide automation tests for NHC Rollups?"
]

for query in queries:
    print("\n" + "=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)
    
    result = rag_chain.query(
        user_query=query,
        top_k=5,
        include_related=True
    )
    
    print(f"\nANSWER:\n{result['answer']}")
    print(f"\nConfidence: {result['confidence']:.1%}")
    print(f"Sources: {len(result['sources'])}")


QUERY: Explain the methodology or approach for testing NHC Rollups test cases.?

Step 1: Retrieving relevant documents...
Generating embeddings for 1 texts...



QUERY: Explain the methodology or approach for testing NHC Rollups test cases.?


Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "H


ANSWER:
Based on the provided context, the methodology or approach for testing NHC Rollups test cases can be outlined as follows:

**1. Test Case Documentation and Preparation**
- Test cases for NHC Rollups are documented in files such as **I-11407 Test Case.docx** and **I-09710 TC.docx**. These documents detail the specific test scenarios, expected outcomes, and steps to be followed during testing.

**2. Test Setup**
- The **Test Setup.docx** provides information on the physical and procedural setup required for executing the tests. This includes:
  - Preparation of the test environment, including hardware and software configurations.
  - Step-by-step instructions for initializing the test system.
  - Reference to related images that visually depict the setup, which can aid in ensuring correct configuration.

**3. Execution of Test Cases**
- The actual testing involves following the steps outlined in the test case documents. This typically includes:
  - Applying specific inputs or co

Embedding batches: 100%|██████████| 1/1 [00:02<00:00,  2.11s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "H


ANSWER:
Based on the provided documentation, the areas affected when any feature is implemented for NHC Rollups are as follows:

1. **Test Data Setup**  
   The document "Test Data Setup.docx" provides details on how test data is prepared for NHC Rollups. Any feature implementation will require updates or changes in test data setup to ensure proper validation and coverage. This includes:
   - Creation and modification of test data specific to NHC Rollups.
   - Ensuring data integrity and consistency for rollup scenarios.
   - Related text entries indicate that multiple aspects of test data setup are involved.

2. **Test Cases**  
   "Test Cases.docx" outlines the test cases associated with NHC Rollups. Feature implementation will impact:
   - The design and execution of test cases for NHC Rollups.
   - Validation of new or changed functionality.
   - Associated images and text in this document may provide visual and descriptive guidance for affected test scenarios.

3. **Location Roll

Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.66s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "H


ANSWER:
Based on the provided context, the document I-11407 Test Case.docx is referenced multiple times and is likely to contain information about test cases, including those related to NHC Rollups. However, the specific details or steps for automation tests for NHC Rollups are not explicitly described in the provided excerpts or summaries.

The Test Coverage Map.xlsx may contain a table mapping test cases to requirements, but no direct content from this file is included in the context.

**Summary:**
- There is no explicit list or description of automation tests for NHC Rollups in the provided context.
- The document I-11407 Test Case.docx is the most relevant source, but without its detailed content, the automation tests cannot be specified.

**Conclusion:**
The context does not contain enough information to provide specific automation tests for NHC Rollups. For a comprehensive answer, please provide the relevant test case details or steps from I-11407 Test Case.docx or another sourc

## Step 7: Custom Query Parameters

In [10]:
# Experiment with different parameters
query = "Summarize the key points from these documents"

print("Testing different retrieval configurations:\n")

# Configuration 1: Few results, no relationships
print("\n1. MINIMAL CONTEXT (top_k=3, no relationships)")
print("=" * 80)
result1 = rag_chain.query(
    user_query=query,
    top_k=3,
    include_related=False
)
print(f"Retrieved: {result1['retrieved_count']} elements")
print(f"Confidence: {result1['confidence']:.1%}")
print(f"\nAnswer length: {len(result1['answer'])} chars")

# Configuration 2: More results with relationships
print("\n\n2. RICH CONTEXT (top_k=10, with relationships)")
print("=" * 80)
result2 = rag_chain.query(
    user_query=query,
    top_k=10,
    include_related=True,
    max_context_length=8000
)
print(f"Retrieved: {result2['retrieved_count']} elements")
print(f"Confidence: {result2['confidence']:.1%}")
print(f"\nAnswer length: {len(result2['answer'])} chars")

print("\n" + "=" * 80)
print("\n✓ RAG system demonstration complete!")


QUERY: Summarize the key points from these documents

Step 1: Retrieving relevant documents...
Generating embeddings for 1 texts...


Testing different retrieval configurations:


1. MINIMAL CONTEXT (top_k=3, no relationships)


Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.10s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
  Retrieved 3 main results
  Retrieved 0 related elements

Step 2: Assembling context...
  Context length: 413 chars

Step 3: Generating answer...
HTTP Request: POST https://initiative.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"

Step 4: Formatting sources...

ANSWER GENERATED
Confidence: 54.50%
Sources: 3


QUERY: Summarize the key points from these documents

Step 1: Retrieving relevant documents...
Generating embeddings for 1 texts...


Retrieved: 3 elements
Confidence: 54.5%

Answer length: 682 chars


2. RICH CONTEXT (top_k=10, with relationships)


Embedding batches: 100%|██████████| 1/1 [00:00<00:00,  2.94it/s]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "H

Retrieved: 30 elements
Confidence: 54.5%

Answer length: 2793 chars


✓ RAG system demonstration complete!


## Step 8: Performance Metrics

In [11]:
import time

# Measure query performance
test_queries = [
    "What is this document about?",
    "What are the main sections?",
    "Are there any images or tables?"
]

print("Performance Test:\n")
print("=" * 80)

total_time = 0
for idx, query in enumerate(test_queries, 1):
    start_time = time.time()
    
    result = rag_chain.query(
        user_query=query,
        top_k=5,
        include_related=True
    )
    
    elapsed = time.time() - start_time
    total_time += elapsed
    
    print(f"\n{idx}. Query: {query}")
    print(f"   Time: {elapsed:.2f}s")
    print(f"   Retrieved: {result['retrieved_count']} elements")
    print(f"   Answer length: {len(result['answer'])} chars")
    print(f"   Confidence: {result['confidence']:.1%}")

print(f"\n\nAverage query time: {total_time / len(test_queries):.2f}s")
print("=" * 80)

print("\n✓ Complete RAG system operational!")
print("\nNext: 06_testing.ipynb - Comprehensive testing suite")


QUERY: What is this document about?

Step 1: Retrieving relevant documents...
Generating embeddings for 1 texts...


Performance Test:



Embedding batches: 100%|██████████| 1/1 [00:02<00:00,  2.07s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
  Retrieved 5 main results
  Retrieved 6 related elements

Step 2: Assembling context...
  Context length: 892 chars

Step 3: Generating answer...
HTTP Request: POST https://initiative.opena


1. Query: What is this document about?
   Time: 5.26s
   Retrieved: 11 elements
   Answer length: 1056 chars
   Confidence: 50.3%


Embedding batches: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/scroll "H


2. Query: What are the main sections?
   Time: 12.30s
   Retrieved: 25 elements
   Answer length: 2329 chars
   Confidence: 73.3%


Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]
✓ Generated 1 embeddings
HTTP Request: POST http://localhost:6333/collections/confidential_docs/points/query "HTTP/1.1 200 OK"
  Retrieved 5 main results
  Retrieved 0 related elements

Step 2: Assembling context...
  Context length: 486 chars

Step 3: Generating answer...
HTTP Request: POST https://initiative.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"

Step 4: Formatting sources...

ANSWER GENERATED
Confidence: 57.87%
Sources: 5




3. Query: Are there any images or tables?
   Time: 3.96s
   Retrieved: 5 elements
   Answer length: 629 chars
   Confidence: 57.9%


Average query time: 7.17s

✓ Complete RAG system operational!

Next: 06_testing.ipynb - Comprehensive testing suite
